# Notebook 02: RAG Pipeline Prototype

**Goal:** Build the full RAG pipeline from scratch in one notebook so you understand every step.

By the end, you will be able to:
1. Embed financial text into vectors
2. Store and retrieve from ChromaDB
3. Pass retrieved context to Claude
4. Understand why prompt caching matters

In [ ]:
import sys
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

print('Environment loaded')

## Step 1: Understand Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model (downloads ~80MB on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed some financial sentences
sentences = [
    'Reliance Industries revenue grew 12% to ₹9.67 lakh crore in FY2024',
    'HDFC Bank net interest margin improved to 4.3% this quarter',
    'TCS reported strong deal wins of $4.5 billion in Q4FY24',
    'The company has high debt levels with debt-to-equity ratio of 2.1',
    'Management raised guidance for next fiscal year citing strong demand',
]

embeddings = model.encode(sentences)
print(f'Embedding shape: {embeddings.shape}')
print(f'Each sentence = {embeddings.shape[1]}-dimensional vector')

In [ ]:
# Semantic similarity — this is how retrieval works
from sklearn.metrics.pairwise import cosine_similarity

query = 'What is the revenue growth of Reliance?'
query_vec = model.encode([query])

similarities = cosine_similarity(query_vec, embeddings)[0]

print(f'Query: "{query}"\n')
print('Similarity scores (higher = more relevant):')
for sent, score in sorted(zip(sentences, similarities), key=lambda x: -x[1]):
    print(f'  {score:.3f}  {sent[:70]}')

# The Reliance revenue sentence should score highest — that's RAG working

## Step 2: ChromaDB — Store and Retrieve

In [ ]:
import chromadb
from chromadb.config import Settings

# In-memory for prototype (switch to PersistentClient for production)
client = chromadb.EphemeralClient(settings=Settings(anonymized_telemetry=False))
collection = client.create_collection('rag_prototype', metadata={'hnsw:space': 'cosine'})

# Add our financial sentences
collection.add(
    documents=sentences,
    embeddings=embeddings.tolist(),
    ids=[f'doc_{i}' for i in range(len(sentences))],
    metadatas=[
        {'ticker': 'RELIANCE.NS', 'section': 'income_statement'},
        {'ticker': 'HDFCBANK.NS', 'section': 'key_ratios'},
        {'ticker': 'TCS.NS', 'section': 'news'},
        {'ticker': 'RELIANCE.NS', 'section': 'key_ratios'},
        {'ticker': 'TCS.NS', 'section': 'earnings_transcript'},
    ]
)

print(f'Documents in collection: {collection.count()}')

In [ ]:
# Retrieve — the core of RAG
def retrieve(query_text, n_results=3, ticker_filter=None):
    q_vec = model.encode([query_text]).tolist()
    kwargs = {'query_embeddings': q_vec, 'n_results': n_results, 'include': ['documents', 'metadatas', 'distances']}
    if ticker_filter:
        kwargs['where'] = {'ticker': ticker_filter}
    results = collection.query(**kwargs)
    return list(zip(results['documents'][0], results['metadatas'][0], results['distances'][0]))

print('Query: "Reliance revenue growth"')
for doc, meta, dist in retrieve('Reliance revenue growth', ticker_filter='RELIANCE.NS'):
    print(f'  Score: {1-dist:.3f} | {meta["section"]:20s} | {doc[:60]}')

## Step 3: Complete RAG Call with Claude

In [ ]:
import anthropic
import os

api_key = os.getenv('ANTHROPIC_API_KEY')
if not api_key:
    print('Set ANTHROPIC_API_KEY in .env first!')
else:
    client_llm = anthropic.Anthropic(api_key=api_key)

    # Retrieve relevant context
    question = 'What is Reliance Industries financial performance?'
    retrieved = retrieve(question, n_results=3)
    context = '\n'.join([f'[{i+1}] {doc}' for i, (doc, _, _) in enumerate(retrieved)])

    print(f'Context passed to Claude:\n{context}\n')

    # Call Claude with context
    response = client_llm.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=500,
        system='You are a concise financial analyst. Answer using ONLY the provided context. Cite specific numbers.',
        messages=[{
            'role': 'user',
            'content': f'Context:\n{context}\n\nQuestion: {question}'
        }]
    )

    print('Claude\'s response:')
    print(response.content[0].text)
    print(f'\nTokens used — input: {response.usage.input_tokens}, output: {response.usage.output_tokens}')

## Step 4: Run the Full Production Pipeline

Now try the actual production modules:

In [ ]:
from src.data_pipeline.pipeline import DataPipeline
from src.rag.ingestion import RAGIngestion
from src.rag.retriever import Retriever

# Fetch + process + embed one ticker
rag = RAGIngestion()
n = rag.ingest('RELIANCE.NS')
print(f'Ingested {n} chunks for RELIANCE.NS')
print(f'Total documents in store: {rag.store.count()}')

In [ ]:
# Test retrieval on real data
retriever = Retriever(vector_store=rag.store)

queries = [
    'What is Reliance debt levels?',
    'Reliance Jio telecom business performance',
    'Reliance free cash flow capex',
]

for q in queries:
    results = retriever.retrieve(q, ticker='RELIANCE.NS', top_k=2)
    print(f'\nQuery: "{q}"')
    for r in results:
        print(f'  [{r["score"]:.3f}] {r["metadata"]["section"]:20s} | {r["text"][:80]}...')